# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — data, connection, Week-4 baseline, and a final deployment model

*Reused verbatim from Weeks 3-6: DuckDB connection, cached aggregation, `feature_df` build, and the Week-4 baseline rule. One addition: a Logistic Regression fit on the full dataset (not held out), used only to produce a per-row confidence score for the playbook — its honest held-out performance was already measured in Week 6 and is not re-claimed here.*

In [ ]:
import os
os.makedirs('skills/writing-honest-claims', exist_ok=True)
os.makedirs('skills/flyrank/flyrank-data', exist_ok=True)
!wget -q -O "skills/writing-honest-claims/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/writing-honest-claims/SKILL.md"
!wget -q -O "skills/flyrank/flyrank-data/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/flyrank/flyrank-data/SKILL.md"
print("skills loaded")


In [ ]:
import duckdb, pandas as pd, os

HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if HF_TOKEN is None:
    from getpass import getpass
    HF_TOKEN = getpass('Enter your Hugging Face token: ')

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print(con.execute(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone())


In [ ]:
max_date = pd.to_datetime(con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0])
last30_start = max_date - pd.Timedelta(days=30)
prev30_start = max_date - pd.Timedelta(days=60)

con.execute(f"""
CREATE OR REPLACE TABLE agg_cached AS
SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_last30,
    AVG(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_last30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
    AVG(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_prev30
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id, content_hash_id
""")
print(con.execute("SELECT COUNT(*) FROM agg_cached").fetchone())


In [ ]:
query = f"""
SELECT
    a.*,
    cl.access_profile, cl.gsc_data_start, cl.ga4_data_start,
    dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
    dc.content_updated_date, dc.is_published, dc.is_deleted,
    DATE_DIFF('day', dc.content_updated_date, DATE '{max_date.date()}') AS days_since_last_update
FROM agg_cached a
JOIN {TABLES['dim_clients']} cl ON a.client_hash_id = cl.client_hash_id
LEFT JOIN {TABLES['dim_content']} dc ON a.content_hash_id = dc.content_hash_id
WHERE a.imp_prev30 >= 100
"""
feature_df = con.execute(query).fetchdf()
feature_df['ctr_prev30'] = feature_df['clk_prev30'] / feature_df['imp_prev30'] * 100
feature_df['is_declining_label'] = feature_df['imp_last30'] < 0.8 * feature_df['imp_prev30']
print(feature_df.shape)


In [ ]:
import numpy as np

bins = [0, 90, 180, np.inf]
labels = ['<=90', '91-180', '181+']
feature_df['staleness_bucket'] = pd.cut(feature_df['days_since_last_update'], bins=bins, labels=labels, right=True)

pos_bins = [-np.inf, 3, 10, 20, 50, np.inf]
pos_labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
feature_df['position_tier_prev'] = pd.cut(feature_df['pos_prev30'], bins=pos_bins, labels=pos_labels, right=True)

median_ctr_by_tier = feature_df.groupby('position_tier_prev', observed=True)['ctr_prev30'].median().rename('median_ctr_prev30').reset_index()
feature_df = feature_df.merge(median_ctr_by_tier, on='position_tier_prev', how='left')

feature_df['weak_ctr_flag'] = (
    (feature_df['ctr_prev30'] == 0) |
    ((feature_df['median_ctr_prev30'] > 0) & (feature_df['ctr_prev30'] < feature_df['median_ctr_prev30']))
)
feature_df['stale_flag'] = (feature_df['days_since_last_update'] > 90).astype(int)
feature_df['baseline_score'] = feature_df['stale_flag'] + feature_df['weak_ctr_flag'].astype(int)
feature_df['baseline_pred'] = (feature_df['baseline_score'] >= 1).astype(int)

def get_reason_code(row):
    if row['stale_flag'] and row['weak_ctr_flag']:
        return 'STALE+WEAK_CTR'
    elif row['stale_flag']:
        return 'STALE'
    elif row['weak_ctr_flag']:
        return 'WEAK_CTR'
    else:
        return 'NONE'

feature_df['reason_code'] = feature_df.apply(get_reason_code, axis=1)
score_to_action_map = {0: 'leave', 1: 'monitor', 2: 'refresh'}
feature_df['action'] = feature_df['baseline_score'].map(score_to_action_map)

print(feature_df['reason_code'].value_counts())


In [ ]:
# Final Logistic Regression, fit on the FULL dataset for deployment scoring.
# Week 6 already measured its honest held-out performance under a grouped
# split (F1 0.771) -- that stays the trustworthy performance record. This
# fit is for producing a confidence score on every row for the playbook,
# which is standard practice for a final deployed model, and is disclosed
# explicitly in Section 2 below rather than presented as a held-out result.
from sklearn.linear_model import LogisticRegression

feature_cols_numeric = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                        'days_since_last_update', 'word_count', 'char_count']
feature_cols_categorical = ['content_type', 'main_intent']

model_df = feature_df.copy()
model_df[feature_cols_numeric] = model_df[feature_cols_numeric].fillna(model_df[feature_cols_numeric].median())
model_df[feature_cols_categorical] = model_df[feature_cols_categorical].fillna('unknown')

X = pd.get_dummies(model_df[feature_cols_numeric + feature_cols_categorical],
                    columns=feature_cols_categorical, drop_first=True)
y = model_df['is_declining_label'].astype(int)

final_lr = LogisticRegression(max_iter=2000, random_state=42)
final_lr.fit(X, y)
feature_df['model_confidence'] = final_lr.predict_proba(X)[:, 1]

print(feature_df['model_confidence'].describe())


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Why the baseline leads the ranking, not the model.** Week 6's honest, grouped-split comparison showed the rule-based baseline had the best precision (0.735) of the three approaches, while Random Forest was actually the *weakest* on F1 (0.695) — worse than the simple rule. A playbook a reviewer has to trust repeatedly needs to lead with the most precise, most explainable signal, not the one with the flashiest metric in isolation. So `reason_code` (from the baseline) drives the primary grouping, and the model's `model_confidence` score is used only as a secondary, within-group ordering signal — never as the headline justification for an action.

**Reason codes, in plain language a non-technical reviewer can trust:**
- `STALE+WEAK_CTR` → **refresh, highest priority.** Hasn't been updated in 90+ days *and* getting fewer clicks than similar pages at its search position. Two independent warning signs.
- `STALE` → **refresh.** Hasn't been updated in 90+ days. Click performance looks normal for its position, but content may be factually outdated.
- `WEAK_CTR` → **monitor.** Reasonably fresh, but getting fewer clicks than similar pages at its position — worth checking the title/snippet or search-intent match.
- `NONE` → **leave.** No warning signs from either check. Lowest priority.

In [ ]:
confidence_bins = [0, 0.33, 0.66, 1.01]
confidence_labels = ['Low', 'Medium', 'High']
feature_df['confidence_tier'] = pd.cut(feature_df['model_confidence'], bins=confidence_bins,
                                        labels=confidence_labels, right=False, include_lowest=True)

reason_severity = {'STALE+WEAK_CTR': 0, 'STALE': 1, 'WEAK_CTR': 2, 'NONE': 3}
feature_df['reason_rank'] = feature_df['reason_code'].map(reason_severity)

ranked_queue = feature_df.sort_values(
    by=['reason_rank', 'model_confidence', 'imp_prev30'],
    ascending=[True, False, False]
).reset_index(drop=True)

display_cols = ['content_hash_id', 'action', 'reason_code', 'confidence_tier',
                'model_confidence', 'days_since_last_update', 'ctr_prev30',
                'position_tier_prev', 'imp_prev30']
print(ranked_queue[display_cols].head(20).to_string(index=False))
print("\nAction distribution:")
print(ranked_queue['action'].value_counts())


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who this is for:** a content or SEO reviewer with limited time, using this queue to decide which pages to look at first — not to decide the outcome automatically.

**What it's valid for:** prioritizing review order across pages that meet the coverage below. Not valid as: proof any individual page is declining, a content-quality score, or a causal explanation for why a page underperforms.

**Coverage limits, restated honestly:**
- Only covers pages with `imp_prev30 >= 100` — the code cell below quantifies exactly how much of the full portfolio that excludes.
- `is_declining_label` is a proxy based on the *current* 30-day window, not a proven future outcome.
- The window anchor (`max_date`) is global, not per-client — clients whose data ends earlier are underrepresented.
- `model_confidence` comes from a model fit on the full dataset for deployment; its honest, held-out accuracy is the Week 6 number (F1 0.771 under a grouped split), not a claim being re-measured here.
- CTR-tier comparisons inherit the volume-floor caveat from the data dictionary: low-traffic tiers can show noisy medians.

In [ ]:
total_content = con.execute(f"SELECT COUNT(*) FROM {TABLES['dim_content']}").fetchone()[0]
covered = feature_df.shape[0]
print(f"Total content items in the warehouse: {total_content}")
print(f"Content items covered by this playbook (imp_prev30 >= 100): {covered}")
print(f"Coverage: {covered / total_content:.1%}")


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any `refresh` or `monitor` recommendation, a reviewer must personally check:**
- Is this actually evergreen or reference content where staleness doesn't matter?
- Is the position tier inherently low-intent traffic, where a lower CTR is normal, not a problem?
- Is the page still live and published? (checked directly below — a deleted or unpublished page should never appear in an active review queue)

**Never automate, under any circumstance:**
- No auto-editing, auto-redirecting, or auto-unpublishing content based on this score alone.
- No using this score as a judgment of a writer's or team's performance.
- No applying this playbook to clients or pages outside the coverage confirmed in Section 2 — it was not validated on them.

In [ ]:
no_go = feature_df[(feature_df['is_deleted'] == True) | (feature_df['is_published'] == False)]
print(f"Rows in feature_df flagged deleted or unpublished: {len(no_go)}")

ranked_queue_clean = ranked_queue[
    (ranked_queue['content_hash_id'].isin(no_go['content_hash_id'])) == False
].reset_index(drop=True)
print(f"Ranked queue rows before no-go filter: {len(ranked_queue)}")
print(f"Ranked queue rows after no-go filter:  {len(ranked_queue_clean)}")


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signals worth rechecking on a schedule, against the reference snapshot exported in Section 5:**
- If the staleness-bucket decline rates (currently ~62% / 73% / 79% across `<=90` / `91-180` / `181+`) shift by a meaningful margin, the staleness threshold itself may need revisiting.
- If the median CTR per position tier moves substantially from the values exported this week, the weak-CTR thresholds are stale.
- Rerun Week 6's honest grouped-split comparison periodically — if the model's F1 drops meaningfully below its Week 6 baseline (0.771) on fresh data, it's time to retrain, not just re-score.
- If portfolio coverage (Section 2's ratio) drops sharply, the `imp_prev30 >= 100` floor may be excluding a growing share of real content, and the threshold should be reconsidered.

In [ ]:
reference_snapshot = {
    'staleness_decline_rate': feature_df.groupby('staleness_bucket', observed=True)['is_declining_label'].mean().to_dict(),
    'median_ctr_by_tier': median_ctr_by_tier.set_index('position_tier_prev')['median_ctr_prev30'].to_dict(),
    'portfolio_coverage': covered / total_content,
}
print(reference_snapshot)


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on these files.*

Three exports: the ranked queue CSV (`work/outputs/`, regenerated every run, not committed), a metrics JSON with the reference snapshot and Week 6's honest comparison numbers (`work/outputs/`, committed — the receipts), and one figure (`work/figures/`, committed) showing the honest-split before/after finding from Week 6, since that's the single most decision-relevant chart for the paper's methodology section.

In [ ]:
import json
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Ranked queue CSV -- regenerated every run, not committed
queue_cols = ['content_hash_id', 'client_hash_id', 'action', 'reason_code',
              'confidence_tier', 'model_confidence', 'days_since_last_update',
              'ctr_prev30', 'position_tier_prev', 'imp_prev30']
ranked_queue_clean[queue_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Wrote work/outputs/action_playbook_queue.csv ({len(ranked_queue_clean)} rows)")

# 2. Metrics JSON -- committed, the receipts
metrics = {
    'coverage': {'total_content': int(total_content), 'covered': int(covered), 'coverage_pct': covered / total_content},
    'reference_snapshot': {
        'staleness_decline_rate': {str(k): v for k, v in reference_snapshot['staleness_decline_rate'].items()},
        'median_ctr_by_tier': {str(k): v for k, v in reference_snapshot['median_ctr_by_tier'].items()},
    },
    'week6_honest_comparison': {
        'baseline':  {'precision': 0.735123, 'recall': 0.548398, 'f1': 0.628178},
        'logistic_regression_grouped': {'precision': 0.668108, 'recall': 0.911104, 'f1': 0.770911},
        'random_forest_grouped':       {'precision': 0.705061, 'recall': 0.685136, 'f1': 0.694955},
    },
    'action_distribution': ranked_queue_clean['action'].value_counts().to_dict(),
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)
print("Wrote work/outputs/playbook_metrics.json")

# 3. Figure -- committed, the Week 6 honest-split finding
fig, ax = plt.subplots(figsize=(7, 4))
models = ['Logistic\nRegression', 'Random\nForest']
before = [0.781425, 0.789583]
after = [0.770911, 0.694955]
x = range(len(models))
width = 0.35
ax.bar([i - width/2 for i in x], before, width, label='Random split (before)')
ax.bar([i + width/2 for i in x], after, width, label='Grouped split (after)')
ax.set_ylabel('F1 score')
ax.set_title('Honest split reveals inflated performance (Week 6)')
ax.set_xticks(list(x))
ax.set_xticklabels(models)
ax.legend()
plt.tight_layout()
plt.savefig('work/figures/honest_split_before_after.png', dpi=150)
plt.show()
print("Wrote work/figures/honest_split_before_after.png")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.